# MSE 2049 - Tensor Networks: Lecture 04 - How to use ITensors?

**Joaquín Peñuela-Parra**

**Department of Mechanical Engineering and Materials Science**

**University of Pittsburgh, Pittsburgh, Pennsylvania 15261, USA**

This tutorial introduces the basic building blocks needed to work with tensors computationally: how to create, index, and manipulate them, and how operations like matrix decompositions (QR, SVD) let us compress and truncate information systematically. We'll then assemble these pieces to construct quantum states as MPS, compare them against exact diagonalization (ED) for small systems, and apply single- and two-site operators to compute physical observables such as expectation values. Along the way, we'll touch on the Hadamard product and see how MPS can even be used to efficiently encode continuous functions — a perspective that connects tensor networks to numerical analysis and quantum-inspired scientific computing. Finally, we'll briefly explore how these operations scale on GPU hardware, highlighting the performance benefits of tensor libraries for larger simulations. 

By the end of this tutorial, you should have the practical toolkit needed to start building and manipulating tensor network states for your own research problems 🤓.

**ITensor documentation:** https://docs.itensor.org/Overview/

# Preliminaries: Installation and loading

To have a self-contained tutorial, we need to start with the installation. For this, you simply need to use the `Pkg` in Julia.

In [ ]:
# Please uncomment the following lines if you have not already installed ITensors:

# using Pkg 
# Pkg.add("ITensors") #Focuses on the core definition and manipulation of individual ITensor objects, index systems, block-sparse/dense data layouts, and basic tensor contractions.
# Pkg.add("ITensorMPS") #Focuses on tensor networks, specifically Matrix Product States (MPS), Matrix Product Operators (MPO), and algorithms like DMRG and TDVP. Requires ITensors.jl as a dependency. 

In [ ]:
using ITensors
using ITensorMPS

It is worth mentioning that you can precompile and speed up ITensors.jl in Julia using the built-in `ITensors.compile()` function with `PackageCompiler.jl` (you probably also need to install this package it with `Pkg`).

**Why?** Julia uses Just-In-Time (JIT) compilation; thus, the compiler must parse, infer types, and generate native machine code before it can actually execute your function. If a function takes only a fraction of a second to run but 5 seconds to compile, your first execution will feel painfully slow.

**Possible problems:** `ITensors.compile()` generates an image with the compiled package inside ` ~/.julia/sysimages/`. This works well if you are working locally. However, the generated image can be hardware-dependent and generate some driver problems when you are running on an HPC cluster (like CRC). This is very annoying when you are using different types of GPUs.

**What can we do:** Well, the simplest solution (what I do) is running the same code in a very simple case before running your real code; that is enough to solve the first compilation. For example, if you want to simulate a system with $100$ sites, run the code at the beginning with only $4$ sites. You can also create multiple images depending on the hardware and load the specific one you need for each case.

# `Index()` and `ITensor()`: How to define tensors? 

Before we can build any tensor network, we need two basic ingredients: indices (the "legs" of a tensor) and ITensors (the tensors themselves, which carry data and know which indices they have).

- **Index()**: Defines an Index which represents a single tensor leg with a fixed dimension. It carries a unique identity (an internal ID tag), so even two indices with the same dimension are treated as distinct unless they are literally the same Index object. **This is what allows ITensor to automatically figure out how tensors should contract**.

- **Itensor()**: Defines a block of data (its storage) together with the set of Index objects labeling each dimension. Unlike a plain array, **an ITensor doesn't care about the order** in which you list its indices

## Example: Defining a matrix

Imagine we want to define the tensor $A_{ij}$ such as $A_{12} = 2,  A_{22} = 3$ and zero everywhere else. Assume both indices are 3-dimensional, i.e. $i = \{1,2,3 \}$ and $j = \{1,2,3, 4\}$.

In [ ]:
i = Index(3) 
j = Index(4)

A = ITensor(i,j) #Tensor with two indexes, it will be a Matrix
A[i=>1, j => 2] = 2 #This is one way to set the elements inside the tensor (it does not matter the order).
A[i=>2, j => 2] = 3 

@show A

### What does it mean that it does not care about the order? 

The element $A_{12}$ means $i = 1$ and $j = 2$, there are two ways to get it:

In [ ]:
@show A[i=>1,j=>2];
@show A[j=>2,i=>1];
@assert A[i=>1,j=>2] == A[j=>2,i=>1]

This is because each index has a unique internal ID tag:

In [ ]:
@show inds(A); #This is how we can get the indexes of A

Indeed, every time that you create an index, it will have a different tag:

In [ ]:
for k = 1:5
    i = Index(3) 
    @show i
end

You can also add a tag to give some information about the index:

In [ ]:
s = Index(3,"dog,spin") #A little bit random to be honest, I have never used it. But useful if you want to define new sites that are not predefined in ITensors.

Defining each element of the matrix is very unpractical; there is another way to do it. Imagine that you already have your matrix defined as a `Matrix{Float64}` object.

In [ ]:
array = [1.0 2.0 3.0 4.0 ; 5.0 6.0 7.0 8.0] #This is a 2x4 Matrix. 

#We need to define two indices, one with dimension 2 (rows) and one with dimension 4 (columns)
i = Index(4) 
j = Index(2)

#Then we just need to use ITensor()

A = ITensor(array,i,j) #This is another way to give vale
@show A 

#or
A = ITensor(array,j, i) #This is another way to give vale
@show A;

Both tensors contain the same information; however, when we print them, only the second one looks like our initial matrix. 

In [ ]:
array

# `randomITensor()`

This function creates an ITensor with normally distributed random elements instead of specific values. To be honest, in three years I have only used this function for the tutorial and maybe some debugging when I was learning how to use GPU.

In [ ]:
i = Index(2); # Index of dimension 2
j = Index(4); # Index of dimension 4
k = Index(3); # Index of dimension 3

A = randomITensor(i,j,k)
println(A)

# `contract()` and `*`: How to contract two tensors?

Contraction is the fundamental operation for combining tensors in a network: any indices shared between two ITensors are automatically summed over, while unshared indices survive into the result. You don't need to specify which indices to contract — ITensor figures this out for you based on matching Index objects.

**Note:** `*` is the standard way to contract two ITensors. `contract()` is useful when you want to be explicit or contract a list of tensors at once (e.g. contract([A, B, C])). It behaves identically to `*` for two tensors, but is handy for writing generic code or contracting many tensors in sequence.

## Product of two matrices. 
Consider the tensor product: $A_{ij}*B_{ik}$ for the following matrices:

In [ ]:
A_matrix = [1.0 2.0 3.0; 4.0 5.0 6.0; 7.0 8.0 9.0]
B_matrix = [1.0 2.0 3.0; 4.0 5.0 6.0; 7.0 8.0 9.0]/10;

i = Index(3)
j = Index(3)
k = Index(3)

A = ITensor(A_matrix,i,j) 
B = ITensor(B_matrix,i,k) 

@show A
@show B

In [ ]:
inds(A) #This is how you get the indices of A

For contraction we just need to use the function `*` or `contract()`

In [ ]:
@show C = A*B; #Use * is how ITensor do the contraction of the common indexes between A and B.
#@show C = B*A; #Same operation.

In [ ]:
@show C = contract(A,B);

Let's try the same without using Itensor, just Julia. In this case we we do not have an index, just matrices. We must be careful when thinking about which index we want to contract, and do the operation that we really want.

It is not just `A_matrix*B_matrix`.

In [ ]:
A_matrix*B_matrix

We want $A_{ij}*B_{ik}$, and $(A*B)_{ik}$ = $A_{ij}*B_{jk}$, which is a very different quantity.

So $A_{ij}*B_{ik}$ = $A^{T}_{ji}*B_{ik}$ = $(A^{T}*B)_{jk}$

In [ ]:
transpose(A_matrix)*B_matrix

We got the same result, but it was harder without ITensor.

## What if we do not have common indices and use `*`?

If we do not have indices to contract, we should expect a higher-order tensor because $A_{ij}*B_{k} = (AB)_{ijk}$

In [ ]:
A = randomITensor(i,j)
B = randomITensor(k)

@show AB = A*B; #Clearly it is not a matrix any more.

# Exercise: Define and multiply 2 tensors. 

Calculate $A_{ij}*B_{jk}$ from two tensors $A_{ij}$ and $B_{jk}$ with $i = \{ 1,2 \}, j = \{ 1, 2, 3, 4\}$, $k = \{ 1, 2, 3\}$, and

- A[i=1,j=1] = 2.0
- A[1,2] = 0.7
- A[1,4] = 1.9
- A[2,2] = -3.0
- A[2,3] = -0.1
- A[2,4] = 0.5
- B[j=1,k=1] = -3.5
- B[1,2] = 0.2
- B[1,3] = 2.7
- B[2,3] = -1.0
- B[3,1] = -0.6
- B[3,3] = 4.2
- B[4,2] = -0.1
- B[4,3] = 1.7

In [ ]:
#TODO:

#A = ....

#B = ....

#C = ....




In [ ]:
@assert isapprox(Array(C, i,k), [-7.0 0.21 7.93; 0.06 -0.05 3.43])

# `Array()`: How to go from an ITensor to a Julia array?

Sometimes when you are testing your code, you need to compare the matrices with what you would do without ITensors (I do it all of the time). So, it is important to know how to move from an ITensor object to a standard array object in Julia. For this, we just need to use `Array()` and be careful about the indices.

Let's take as an example the tensor built in the previous exercise  $C_{ik} = A_{ij}*B_{jk}$. Assuming that the first index of each tensor refers to the row and the second one to the columns, we would expect $C_ik$ to be a $2 \times 4$ matrix.

In [ ]:
Array(C, i, k) #row, column

If you are not careful about this, you can get the transpose:

In [ ]:
Array(C, k, i)

Clearly saying that one index is a column or a row is a convention; the point here is that **you need to be consistent**. Use the same convention that you are using in your standard linear algebra calculations.

This also works if you have tensors with more than one index. 

In [ ]:
i = Index(2); # Index of dimension 2
j = Index(4); # Index of dimension 4
k = Index(3); # Index of dimension 3

A = ITensor(i,j,k);  # Define ITensor element by element

A[i=>1, j=>1, k=>3] = 2.0;
A[i=>1, j=>2, k=>3] = 0.7;
A[i=>1, j=>1, k=>3] = -3.5;
A[i=>1, j=>2, k=>3] = 0.2;

@show A;

## **Do you have a collection of 3 matrices of size 2x4, or a collection of 4 matrices of size 2x3?**

In [ ]:
Array(A, i, j, k)

In [ ]:
Array(A, i, k, j)

Well, technically both options are correct. Only you know what your simulation is.

## Exercise: Get a collection of 2 matrices with size 4 x 3

In [ ]:
#TODO:

#Collection = ...





In [ ]:
@assert Collection.size == (4,3,2)

# Linear combinations of ITensors

ITensors may also be subtracted and multiplied by scalars, including complex scalars, for example:

In [ ]:
A = randomITensor(i,j,k)
B = randomITensor(k,i,j) 

In [ ]:
@show C = 4*A - B/2 
@show D = A + 3.0im * B;

This is just possible because A and B have the same indexes. Let's see what happens if this is not the case:

In [ ]:
l = Index(3)

A = randomITensor(i,j,k)
B = randomITensor(k,i,l) 

In [ ]:
C = 4*A - B/2 

We already see that we can use * to contract the common indexes between two or more tensors. However, * can be used in different ways if we also use the functions prime(), delta(), combiner() and dag(). Let's go one by one.

# prime(): level of a tensor's indices

Since indices are matched by identity (not just by dimension), sometimes we need two versions of "the same" index that ITensor treats as different — for example, when applying an operator to a state, the input and output physical indices must be distinguishable, even though they represent the same physical site with the same dimension. `prime()` solves this by adding a prime level to an index, effectively creating a labeled copy that has the same dimension and tags but a different identity for contraction purposes.

Consider two Tensors:

In [ ]:
A = randomITensor(i,j)
B = randomITensor(i,j);

## How to get the common indices between different tensors?

In [ ]:
commoninds(A,B) #It returns the id of the common indixes between A and B

As we expected, these indixes are i and j:

In [ ]:
i,j

Then if we use * between A and B, the operator will contract both indexes:

In [ ]:
@show C = A*B; #We got a tensor of range zero (an scalar)

This has sense because it is just $A_{i,j}*B_{i,j}$. IF we contract i and j, then we must have and scalar. We can access to this scalar in two ways:

In [ ]:
C[], scalar(C)

Some cases require us not to contract over all common indices. 

## Contracting only over one index even if there are two common indices

To avoid contraction over one common index (say $i$), we need to prime it in one of the tensors:

In [ ]:
A_prime = prime(A,i);

The ITensor A_prime has the same elements as A but has indices (i',j) instead of (i,j).

In [ ]:
@show A
@show A_prime;

You can also use `prime()` directly on an index object:

In [ ]:
prime(i), prime(prime(i))

In case that you want to see how many primes you have you can use `plev()`:

In [ ]:
plev.(inds(A)) #Here I am using that . in Julia means to apply to all elements of the list, and inds(A) gives a list with the indices of A

In [ ]:
plev.(inds(A_prime))

In [ ]:
plev(prime(i))

You can also restart the prime level to zero by using `noprime()`:

In [ ]:
i'', noprime(i'')

We can also prime all indices of the tensor at the same time rather than just one:

In [ ]:
@show prime(A,i,j); #Same elements as A but with indices (i',j') instead of (i,j). 

In [ ]:
@show A'; #Another way to do it.

In this case A_prime and B just have one common index:

In [ ]:
commoninds(A_prime,B)

So,

In [ ]:
@show C = A_prime*B;

Instead of doing $A_{i,j}*B_{i,j}$, we did $A_{i',j}*B_{i,j} = C_{i',i}$. 

Now if we want to get the same scalar as before, we just need to sum elements where $i = i'$. This is equivalent to multiplying by the Kronecker delta tensor.

# delta(): Identity or Kronecker delta tensor

This function creates a Kronecker delta tensor: a tensor that is 1 when all its indices take the same value, and 0 otherwise. It's the tensor-network equivalent of an identity map.

In [ ]:
@show delta(i,i');

Then,

In [ ]:
@show C*delta(i,i');

In general, delta() output is the identity tensor, we can build it with more indexes:

In [ ]:
@show delta(i,j,k);

Indeed, Julia allows us to use the symbol `δ` directly instead of delta.

In [ ]:
@show δ(k,i,j) == delta(i,j,k);

## How to replace indices?

Sometimes we want to change one index for another one. Let's say that you define two matrices with different indices, but you want to multiply them. For this, they need a common index. We can exchange indices with the identity, simply noting that $A_{j} = \sum_{i} B_{i}\delta_{i,j}$. Usually, thi|s is done considering the same dimension.

In [ ]:
i = Index(3); # Index of dimension 2
j = Index(3); # Index of dimension 2
@show i,j

A = randomITensor(i)
@show A;

A = A*delta(i,j)
@show A;

## Tracing out some degrees of freedom.

We could think that maybe we can split the legs of the tensor using deltas, but it is dangerous. Consider you have $A_{i}$ and you want to reshape it as a matrix $A_{kl}$.

In [ ]:
i = Index(2); # Index of dimension 2
j = Index(2); # Index of dimension 2
m = Index(4)
@show A = randomITensor(m); 

You could multiply with a `delta(m,j,k)` and it will become a matrix, but **it will not necessarily have all the information**. The right process for reshaping the matrix should use the `combiner()` function explain in the next section.

In [ ]:
@show A*delta(m,j,k);

This property of "losing" information is very useful to trace out freedom degrees (a very common operation in quantum mechanics). One example of performing the partial trace in quantum mechanics can be found here: https://itensor.discourse.group/t/trace-and-partial-trace-of-mpo/79/2

In conclusion, `delta()` can be very useful but you need to be careful about how you want to use it.

In [ ]:
A = randomITensor(i,j,l); #Suppose that we want to trace out j, l and just keep i.
@show A*delta(j,l); # Very useful but also dangerous.

# Exercise: Contract over $_{i}$ in $A_{ij} B_{ji}$ and get a rank-1 tensor.

We want $C_{j} = \sum_{i} A_{ij} B_{ji}$. From implementing the definition this would be:

In [ ]:
i = Index(3); # Index of dimension 2
j = Index(4); # Index of dimension 4

A = randomITensor(i,j)
B = randomITensor(j,i)

In [ ]:
# You could use just the definition, but that is not the point of the exercise:

C_from_definition = ITensor(j)

for j_index = 1:dim(j)
    C_from_definition[j => j_index] = 0.0
    for i_index = 1:dim(i)
        C_from_definition[j => j_index] += A[i => i_index, j => j_index]*B[j => j_index, i => i_index]
    end
end

@show C_from_definition;

However, the idea of the exercise is to use `prime()` and `delta()` to get the same tensor. 

- **Hint 1:** You need to use primes to avoid contraction over $j$. 
- **Hint 2:** $A_{ij} B_{j'i}$ gives a tensor with indices $j,j'$ and $C$ has only one index. 
- **Hint 3:** You need to define a new index by analyzing $j' = j$. You can do this multiplying by  $\delta_{j,j',j''}$.

In [ ]:
#TODO:

# C = ...





In [ ]:
@assert isapprox(C, C_from_definition)

# Exercise: Contract over $_{i,j}$ in $A_{ij} B_{jk} C_{km} D_{mi}$ and get a rank-2 tensor.

We want $F_{km} = \sum_{i,j} A_{ij} B_{jk} C_{km} D_{mi}$

In [ ]:
#TODO:

i = Index(2) # Index of dimension 2
# j =  ... # Index of dimension 4
# k =  ... # Index of dimension 2
# m =  ... # Index of dimension 4

A = randomITensor(i,j) #
# B = ... #
# C = ... #
# D = ... #;

In [ ]:
F_from_definition = ITensor(k, m)

for k_index = 1:dim(k)
    for m_index = 1:dim(m)
        F_from_definition[k => k_index, m => m_index] = 0.0
        for i_index = 1:dim(i)
            for j_index = 1:dim(j)
                F_from_definition[k => k_index, m => m_index] += A[i => i_index, j => j_index]*B[j => j_index, k => k_index]*C[k => k_index, m => m_index]*D[m => m_index, i => i_index]
            end
        end
    end
end

In [ ]:
#TODO:

# F = ...





In [ ]:
@assert isapprox(F, F_from_definition); #No sure if it was simpler to be honest haha.

Maybe right now `prime()` doesn't look very useful, but actually it is. For example, it can be used to create density matrix operators from quantum states without contracting all the indices in the process.

# `combiner()`: How to mix tensor legs?

Sometimes we need to merge several indices into a single one (or split one index back into several) — for example, turning a rank-3 tensor $A_{ijk}$ into a matrix $A_{(ij),k}$ before doing an SVD.

In [ ]:
i = Index(3); # Index of dimension 2
j = Index(4); # Index of dimension 4
k = Index(3); # Index of dimension 2
l = Index(5); # Index of dimension 4

A = randomITensor(i,j,k,l);

We have a tensor of order 4. If we want to reshape it as a tensor of order 2, we can use the function combiner to merge $i, j$ and $k$ indices:

In [ ]:
@show Mix_i_j_k = combiner(i,j,k, tags = "ijk"); #tags is an optional parameter when we define indexes, is just a label to know what means the index. In this case is useful to remember which indexes was merged.

Then we just need to do the product:

In [ ]:
@show A_reshaped_ijk_l = Mix_i_j_k*A;

## Exercise: Combine $i,j$ and $k,l$ to have a matrix of dimensions dim(i)*dim(j) x dim(k)*dim(l). 

For this, use the same A as before. 

**Hint:** You need to define two combiners.

In [ ]:
#TODO:

# A_reshaped_ij_kl = ...





In [ ]:
@assert dims(A_reshaped_ij_kl) == (dim(i)*dim(j), dim(k)*dim(l))

**What about reshaping a 4-dimensional index back into two 2-dimensional ones?** This is a reshape, and unlike `delta()` (which only picks out a diagonal and loses information), `combiner()` can be used to perform a genuine, information-preserving reshape. However, in that scenario `combiner()` is not the whole story, we need another function.

# `dag()`: How to split tensor legs? & How to get adjoint operators?

This function (short for "dagger") returns the complex conjugate of an ITensor's elements. 

**What does this have to do with splitting legs and reshaping?** A combiner can be viewed mathematically as a rectangular isometric matrix C, i.e., combining indices (mix the legs) behaves like multiplying by C. Therefore, to undo that multiplication (split the legs), you must multiply by its inverse or adjoint, $C^{\dagger}$.

To recover the initial shape we jused need to do the following:

In [ ]:
A_reconstructed_1 = dag(Mix_i_j_k)*A_reshaped_ijk_l
A_reconstructed_2 = dag(Mix_k_l)*(dag(Mix_i_j)*A_reshaped_ij_kl)

# Let's see if these tensor are the same as A:
@show A == A_reconstructed_1 == A_reconstructed_2;

With this in mind, let's go back to the example where `delta()` did not work to reshape the matrices:

In [ ]:
m = Index(4)
@show A = randomITensor(m); 

In [ ]:
i = Index(2); # Index of dimension 2
j = Index(2); # Index of dimension 2
@show A*delta(m,j,k);

In [ ]:
C = combiner(i, j) # combiner: (k,l) <-> one combined index of dim 4
ci = combinedind(C) # this is the dim-4 combined index.
A = A*delta(m, ci) #We need to change the index of our matrix to the combined index
@show A*dag(C) #Now we apply the dagger operation to split the combined index.

Now, let's start considering complex numbers to see that this function is actually performing the dagger.

In [ ]:
@show A = randomITensor(i) + 1im*randomITensor(i) # This creates a complex tensor; at the end is a linear combination using a complex scalar in the sum. 1im = i in Julia.

In [ ]:
@show dag(A)

Remember that **tensors do not have a notion of inputs and outputs; they only contain indices that can be contracted with other tensors**. Remember that `dag()` applies the conjugate to all tensors; we still need to decide which index is the input and which is the output of the operator. 

Consider a 2x3 matrix.

In [ ]:
A_matrix = [1.0 2.0im 3.0 + 2im; 4.0im 5.0 6.0im] 

j = Index(3) #input
i = Index(2) #output
A = ITensor(A_matrix,i,j) 

In [ ]:
@assert Matrix(A, i,j) == Array(A, i,j); #This is another way to go from tensors to standard julia objects.
Matrix(A, i,j) 

In [ ]:
A[i=>1,j=>2], A_matrix[1, 2]

In [ ]:
A_matrix_dag = adjoint(A_matrix)

In [ ]:
Adag = dag(A)

In [ ]:
Matrix(Adag, i, j) #Always the first index that you put on Matrix will correspond to columns and the second to rows. You can think of each row as a collection of columns. 

In [ ]:
Matrix(Adag, j, i)

In [ ]:
Adag[i=>2,j=>3], A_matrix_dag[3,2]

Let's see why is important to keep this in mind. Sometimes in the research you would need to implement in the tensor networks language an operation that in math notation looks like $A^{\dagger}A$ (I do it all the time for open quantum systems). This means that the output of $A$ should be contracted with the input of $A^{\dagger}$.

In [ ]:
adjoint(A_matrix)*A_matrix

In [ ]:
Adag = dag(A)
Adag = Adag*delta(j, j') #We just want to contract the output of A (i) with the input of Adag (i). We need to rename the output of Adag in order to avoid that contraction. 
Matrix(A*Adag, j, j')

## Exercise: Implement $AA^{\dagger}$

In [ ]:
#TODO:

# A_Adag = ...





In [ ]:
@assert isapprox(A_matrix*adjoint(A_matrix), Matrix(A_Adag, i, i'))

In conclusion, when you use `dag()` think about what you really want to contract. 

# `qr()` and `svd()`: Descomposition of ITensors

QR and SVD are two of the most useful and well-known ways to decompose a matrix (rank-2 tensor):

$$    M = QR, \qquad M = U S V^\dagger $$

where $Q$ is orthogonal ($Q^\dagger Q = \mathbb{1}$) and $R$ is upper-triangular for QR, while $U,V$ are orthogonal/unitary and $S$ is diagonal with non-negative singular values for SVD.

In principle, if we have a higher-rank tensor with indices $i,j,k,l$, decomposing would consists on the following steps: 

- Reshape it into a matrix by grouping indices into a row group and a column group.
- Apply the matrix decomposition
- Reshape the result back so that the output tensors carry the correct original indices.

Fortunately, **qr()** and **svd()** do all of this internally for us.

**All we need to do is tell the function which indices belong to the ``left'' side (i.e.\ which indices should end up on $Q$, or on $U$) --- every other index is automatically grouped on the other side.**

In [ ]:
i = Index(3); # Index of dimension 2
j = Index(4); # Index of dimension 4
k = Index(3); # Index of dimension 2
l = Index(5); # Index of dimension 4

A = randomITensor(i,j,k,l);

In [ ]:
Q,R = qr(A, (i,j)); #Q will have the indices (i,j)
@show Q;

## Exercise: Check that ITensors does all the mixing/splitting for us for `qr()`

In [ ]:
#TODO:

# Q_longprocess = ...
# R_longprocess = ...













@assert isapprox(Q_longprocess, Q)
@assert isapprox(R_longprocess, R)

Similary for `svd()`

In [ ]:
U,S,V = svd(A, (i,j)); #U will have the indices (i,j) 
@show U
@show S
@show V

Let's practice a little bit more

In [ ]:
i = Index(3, "i")
j = Index(3, "j")
k = Index(3, "k")

A = randomITensor(i,j,k);

In [ ]:
Q,R = qr(A,(i,j))

@show inds(Q) #Has i,j
@show inds(R); #Has k

or 

In [ ]:
Q,R = qr(A,(i))

@show inds(Q) #Has i
@show inds(R); #Has j,k

In any case,

In [ ]:
A ≈ Q*R

With SVD descomposition is exactly the same:

In [ ]:
U,S,V = svd(A,(i,j))

@show inds(U) # Has i,j
@show inds(S) 
@show inds(V); # Has k

In [ ]:
U,S,V = svd(A,(i))

@show inds(U) # Has i
@show inds(S) 
@show inds(V); # Has j,k

In any case,

In [ ]:
 U*S*V ≈ A # true

## What about the reduced SVD and the truncation error?

In [ ]:
i = Index(10)
j = Index(40)
k = Index(20)

A = randomITensor(i,j,k) #Is very very big.

In [ ]:
U, S, V = svd(A, (i,k));
@show error = (norm(U*S*V - A)/norm(A))^2; #it contains all the singular values
singular_values = diag(Matrix(S, inds(S)));

When performing an SVD, we often don't want to keep all singular values — especially for large systems, where the bond dimension can grow exponentially. maxdim and cutoff are the two main knobs used to control this truncation.

## Maxdim - the maximum number of singular values (bond dimension) to keep

In [ ]:
U, S, V = svd(A, (i,k), maxdim=5); #We are truncating and getting a reduced SVD with just 5 singular values
@show S
@show error = (norm(U*S*V - A)/norm(A))^2;

In [ ]:
U, S, V = svd(A, (i,k), maxdim=10); #We are truncating and getting a reduced SVD with just 10 singular values
@show S
@show error = (norm(U*S*V - A)/norm(A))^2;

## Cutoff - discards singular values whose relative contribution to the total weight falls below this threshold

In [ ]:
U, S, V = svd(A, (i,k), cutoff = 0.1); #It will take the necessary singular values to have an error of 0.1
@show inds(S) #the number of singular values is the dimension of the index.
@show error = (norm(U*S*V - A)/norm(A))^2;

## Exercise: Calculate the truncation error

Check that the last `(norm(U*S_10*V - A)/norm(A))^2` printed is equivalent to calculate $\cfrac{\sum_{i = \chi + 1}^{N} \lambda_{i}^{2}}{\sum_{i = 1}^{N} \lambda_{i}^{2}}$ with the corresponding number of singular values. For this, you can use the `singular_values` variable defined from the exact SVD.

In [ ]:
#TODO:





This reveals the square root of the Frobenius norm is ignored by ITensors. This is the reference value that ITensors uses for truncation when you use `cutoff`.

## What if we have maxdim and cutoff at the same time?

When we have both `maxdim` and `cutoff`, the decomposition will the necessary singular values to have an error of `cutoff` and then take just take a subset of `maxdim` singular values. Clearly, in this case the error could be bigger than `cutoff`. 

In [ ]:
U, S, V = svd(A, (i,k), maxdim = 5, cutoff = 0.1); #It will take the necessary singular values to have an error of 0.1 and then take just 5 singular values, the error could be bigger than 0.1. 
@show inds(S) #the number of singular values is the dimension of the index.
@show error = (norm(U*S*V - A)/norm(A))^2;

**Note:** Most of tensor networks require using `svd()` internally, so most of the functions receive also `maxdim` and `cutoff` as parameters (more on this later).

# Matrix Product States (MPS)

Now that we know how to decompose tensors with SVD and control truncation, we're ready to tackle the real motivation behind all of this: representing many-body quantum states efficiently. A generic quantum state of $M$ sites, each with local dimension $d$, is described by a rank-$M$ tensor with $d^M$ coefficients — a number that grows exponentially and quickly becomes intractable, even for modest $M$. A **Matrix Product State (MPS)** rewrites this single, unwieldy tensor as a **chain of smaller tensors**, one per site, connected by the bond indices we saw appear naturally out of SVD:

$$        | \psi \rangle = \sum_{n_{1}, \cdots, n_{M} = 1}^{d}  C_{n_{1}, \cdots, n_{M}} | n_{1}, \cdots, n_{M} \rangle $$

$$ C_{n_1, \dots, n_M} = \sum_{a_1=1}^{r_1} \sum_{a_2=1}^{r_2} \sum_{a_3=1}^{r_3} \dots \sum_{a_{M-1}}^{r_{M-1}} 
        A_{a_1}^{n_1}A_{a_1, a_2}^{\sigma_2} A_{a_2, a_3}^{\sigma_3} \dots A_{a_{M-2}, a_{M-1}}^{\sigma_{M-1}} A_{a_{M-1}}^{n_M}$$ 

Instead of storing $d^M$ numbers, we now only need $O(M d \chi^2)$, where $\chi$ is the bond dimension — the very quantity we learned to control via `maxdim` and `cutoff`. In this section, we'll construct MPS explicitly — by hand and by decomposing a full state vector via repeated SVDs — and compare against exact diagonalization (ED) for small systems, so you can see directly how the MPS approximation behaves as we vary the bond dimension and truncation error.

In diagram notation:

<div align="center">

<img src="Figures/MPS.png" width="400"/>

</div>

The first step in defining an MPS is creating a set of indices for the legs. 

In [ ]:
i = Index(2)
j = Index(2)
k = Index(2)
l = Index(2)
m = Index(2)

C = randomITensor(i,j,k,l,m) # From ITensor

In [ ]:
cutoff = 1E-20
maxdim = 10
ψ = MPS(C,(i,j,k,l,m) ;cutoff=cutoff,maxdim=maxdim)

In [ ]:
@show linkdims(ψ); #List with all bond dimension.
@show maxlinkdim(ψ); #Maximum bond dimension.

If we want to recover $C$ we just need to contract all tensors inside $\psi$. We could do it by multiplying one by one with `*`, or we could simply use `contract()`

In [ ]:
@assert isapprox(contract(ψ), C)

Just for pedagogical purposes, in principle, we could also build the MPS doing the SVDs by ourselves and storing the tensors at the end.

In [ ]:
# --- Step 1: split off site i ---
U1, S1, V1 = svd(C, (i,); cutoff=cutoff, maxdim=maxdim) # U1 has indices (i, bond1) -> this is our first MPS tensor, A1
A1 = U1
bond1 = commonind(U1, S1)
R1 = S1 * V1   # # absorb S1 into the remainder so we can keep decomposing. indices: (bond1, j, k, l, m)

# --- Step 2: split off site j ---
U2, S2, V2 = svd(R1, (bond1, j); cutoff=cutoff, maxdim=maxdim)
A2 = U2                     # indices: (bond1, j, bond2)
bond2 = commonind(U2, S2)
R2 = S2 * V2                # indices: (bond2, k, l, m)

# --- Step 3: split off site k ---
U3, S3, V3 = svd(R2, (bond2, k); cutoff=cutoff, maxdim=maxdim)
A3 = U3                     # indices: (bond2, k, bond3)
bond3 = commonind(U3, S3)
R3 = S3 * V3                # indices: (bond3, l, m)

# --- Step 4: split off site l ---
U4, S4, V4 = svd(R3, (bond3, l); cutoff=cutoff, maxdim=maxdim)
A4 = U4                     # indices: (bond3, l, bond4)
bond4 = commonind(U4, S4)
R4 = S4 * V4                # indices: (bond4, m)

# --- Step 5: last site, nothing left to split ---
A5 = R4                     # indices: (bond4, m)

In [ ]:
ψ_from_definition = MPS(ITensor[A1,A2,A3,A4,A5])

In [ ]:
norm(ψ_from_definition - ψ)

To be honest, I have **NEVER** done this long process, haha. It is better to just use `MPS()`, but I thought that it was worth seeing once in your life during the tutorial.

## `siteinds()`: "SiteTypes" to represent different physical degrees of freedom provided by ITensors.

When building an MPS "by hand" as we just did, every index was created manually and had no notion of what kind of physical system it represented — it was just an abstract dimension-2 leg. In practice, this becomes a real burden as soon as you want to apply operators, since you'd need to build every operator matrix (Pauli matrices, raising/lowering operators, number operators...) yourself and keep track of index conventions consistently across the whole chain. 

The `siteinds()` function solves this by creating indices that are aware of the local physical Hilbert space they represent — for example, siteinds("S=1/2", N) creates $N$ spin-1/2 indices, while siteinds("Boson", N; dim=...) creates bosonic indices with a chosen truncation.

In [ ]:
N = 10; # Number of sites
sites = siteinds("Qubit", N) # Indices for all sites

This means that ITensors is considering the next local basis for building states and operators:

$$| 0 \rangle = \begin{pmatrix} 1 \\ 0 \end{pmatrix} \quad , \quad | 1 \rangle = \begin{pmatrix} 0 \\ 1 \end{pmatrix}$$

Let's see another example of `SiteType`.

In [ ]:
siteinds("Electron", N) # Indices for all sites

Now the dimension of each index is 4 instead of 2. This is because ITensors is considering a local basis
$$|0\rangle = \begin{pmatrix} 1 \\ 0 \\ 0 \\ 0 \end{pmatrix}, \quad 
|\uparrow\rangle = \begin{pmatrix} 0 \\ 1 \\ 0 \\ 0 \end{pmatrix}, \quad 
|\downarrow\rangle = \begin{pmatrix} 0 \\ 0 \\ 1 \\ 0 \end{pmatrix}, \quad 
|\uparrow\downarrow\rangle = \begin{pmatrix} 0 \\ 0 \\ 0 \\ 1 \end{pmatrix}$$

However, let's focus on the qubit scenario.

## Defining states from ket notation.

Let's say that we want to build the state $|0000000000 \rangle$ for a qubit system with 10 sites.

In [ ]:
ket_state = ["0" for n=1:N]; 
ψ = MPS(sites, ket_state)

## `expect()`: Calculating expectation values.

To check that this is the right state we can calculate $\langle Z_{i} \rangle$ for each site. We would expect $\langle Z_{i} \rangle = 1 \forall i$.

In [ ]:
@show expect(ψ, "Z");

## Comparison with exact diagonalization objects

In principle, we could also build the exact state from just standard linear algebra, and load it to ITensors.

In [ ]:
zero = [1; 0]
one = [0; 1]

ψ_ED = zero
for i = 1:N-1
    ψ_ED = kron(zero, ψ_ED)
end

@show ψ_ED;

In [ ]:
function Build_Z_j(N::Int64, j::Int64) #Builds Z_j as I⊗...⊗I⊗Z⊗I...⊗I for a system of N sites.
    Z = [1 0; 0 -1]
    Identity = [1 0; 0 1]
    
    M = Identity
    j == 1 ? M = Z : nothing
    for i=2:N 
        i == j ? M = kron(Z, M) :  M = kron(Identity, M) #It is more efficient to save everything as sparse matrices. 
    end

    return M
end

Z_i = [adjoint(ψ_ED)*Build_Z_j(N,i)*ψ_ED for i = 1:N];
@show Z_i;

Ok, so it is the same object. Let's try to load it to ITensors.

In [ ]:
ψ_fromED = MPS(ψ_ED, sites, cutoff = 1e-15)

In [ ]:
@show expect(ψ_fromED, "Z");

Ok, everything is consistent. We could also contract our initial MPS implementation and compare with the ED vector.

In [ ]:
ψ_contracted = contract(ψ) 

In [ ]:
comb = combiner(sites) #Our ED object has only one index; we need to combine all sites into just 1.
@assert Array(ψ_contracted*comb, combinedind(comb)) == ψ_ED

## Exercise: Define $|0,0,0,0,0,1,1,1,1,1 \rangle$ in MPS, check $\langle Z_{i} \rangle$ and compare with the ED object.

In [ ]:
# ED case: Try both cases, but only the first one gives the same expectation values, why?
ψ_ED = kron(one, one, one, one, one, zero, zero, zero, zero, zero); 
# ψ_ED = kron(zero, zero, zero, zero, zero, one, one, one, one, one); 

Z_i = [adjoint(ψ_ED)*Build_Z_j(N,i)*ψ_ED for i = 1:N];
@show Z_i;

In [ ]:
#TODO:
# ψ = ...
# ψ_contracted = ...
# comb = ... 








@assert Array(ψ_contracted*comb, combinedind(comb)) == ψ_ED

This reveals that ITensors, internally, when building the states, is thinking of multiplying the tensor products to the left, rather than to the right as you would do by hand.

## Exercise: Define $|0,1,0,1,0,1,0,1,0,1 \rangle$ in MPS, check $\langle Z_{i} \rangle$ and compare with the ED object.

In [ ]:
#TODO:
# ψ = ...
# ψ_contracted = ...
# comb = ...







@assert Array(ψ_contracted*comb, combinedind(comb)) == ψ_ED

## `randomMPS()`

This function construct a real, random Matrix Product State (MPS) with a specified link (bond) dimension.

In [ ]:
ψ1 = randomMPS(sites,linkdims=10);
ψ2 = randomMPS(sites,linkdims=10);
ψ3 = randomMPS(sites,linkdims=10)

## Addition of MPSs

A natural operation we'd like to do is add two MPS together, e.g. $|\psi\rangle = |\phi_1\rangle + |\phi_2\rangle$ (this comes up constantly — building superpositions, applying a sum of operators, Trotterized time evolution, etc.). But there's an important catch: the bond dimension of the sum is (at most) the sum of the bond dimensions of the two inputs, not their maximum.

In [ ]:
ψ = ψ2 + ψ3 # A cutoff of 1e-15 is used by default
@show linkdims(ψ);

In [ ]:
ψ = +(ψ2, ψ3; cutoff = 1e-8); # Set the cutoff
@show linkdims(ψ);

In [ ]:
ψ = +(ψ2, ψ3; maxdim = 10); # Set maxdim
@show linkdims(ψ);

## `norm()`

This function computes the norm of a tensor or MPS, i.e., $\sqrt{\langle\psi|\psi\rangle}$.

In [ ]:
@show norm(ψ);

In [ ]:
ψ_normalized = ψ/norm(ψ)
@show norm(ψ_normalized);

ψ_normalized = normalize(ψ)
@show norm(ψ_normalized);

normalize!(ψ) #A lot of functions receive the ! and perform an in-place operation, be careful with this.
@show norm(ψ);

## `truncate()`

To truncate an MPS, you do not need to do the `svd()` yourself; you can just call this function.

In [ ]:
ψ_truncated = truncate(ψ, cutoff = 1e-1)
@show linkdims(ψ_truncated);

In [ ]:
ψ_truncated = truncate(ψ, maxdim = 1)
@show linkdims(ψ_truncated);

## Exercise: What is the bond dimension used for a cutoff of 0.02?

In [ ]:
#TODO:








## `inner()`, `op()` and `apply()`: Expectation values and overlaps of MPS

Two of the most common quantities we compute with MPS are overlaps between states, $\langle\phi|\psi\rangle$, and expectation values of operators, $\langle\psi|\hat{O}|\psi\rangle$ — both of which are handled efficiently without ever constructing the full state vector. `inner(φ, ψ)` computes the overlap $\langle\phi|\psi\rangle$ by contracting the two MPS site-by-site (automatically applying `dag()` to φ internally), while `op("OpName", sites[n])` builds a local operator (e.g. "Sz", "Sx", "N") acting on the physical site index `s[n]`, ready to be applied to the MPS with `apply()`. Combining the three lets us compute expectation values cleanly: apply the operator to a copy of the state, then take the overlap with the original. 

In [ ]:
inner(ψ_truncated,ψ) #Check what happens if you change cutoff or maxdim in the previous lines.

In [ ]:
dot(ψ_truncated,ψ) #This gives the same result. It exists mainly for consistency with Julia's LinearAlgebra. However, I prefer inner because it is also compatible with products like <Φ|H|Φ> if H is an MPO.

Now, let's define an apply some operators.

Assume we start from $|0,1,0,1,0,1,0,1,0,1 \rangle$ 

In [ ]:
ket_state = ["0", "1", "0", "1", "0", "1", "0", "1", "0", "1"]; 
ψ = MPS(sites, ket_state);

In [ ]:
@show op("Z", sites[1]);

In [ ]:
Z1 = op("Z", sites[1])
commoninds(ψ[1], Z1) #common indices are inputs and prime indices are outputs.

To act the operator we need to use `apply()`

In [ ]:
ψ_afterZ1 = apply(Z1, ψ) #This also can receive maxdim and cutoff as aprameters.

In [ ]:
@show inner(ψ, ψ_afterZ1) 
@show inner(ψ, apply(Z1, ψ));

Zi = [inner(ψ, apply(op("Z", sites[i]), ψ)) for i = 1:N];
@show Zi;

We could think on applying Hadarmards gates and then check $\langle X_{i} \rangle$. Remember that $H |0 \rangle = | + \rangle $ and $H |1 \rangle = |- \rangle$.

In [ ]:
for i = 1:N
    ψ = apply(op("H", sites[i]), ψ)
end

Xi = [inner(ψ, apply(op("X", sites[i]), ψ)) for i = 1:N];
@show Xi;

## Exercise: Define $| 0 0 1 1 0 0 \rangle$, apply a rotation of $\theta = \pi/2$ at all sites and calculate $\langle Y_{i} \rangle$

Remember the Bloch sphere:

<div align="center">

<img src="Figures/Bloch sphere.png" width="400"/>

</div>

**Hint:** You need to define the operator "Rx" which also takes θ as an argument (the rotation angle around the x-axis).

In [ ]:
#TODO:









# Yi = ...

In [ ]:
@assert isapprox(Yi, [-1, -1, 1, 1, -1, -1])

## von-Neumann entanglement entropy

We've already seen that the SVD singular values $S_\alpha$ carry physical meaning — they control how much of the state's structure survives truncation at a given bond. The von Neumann entropy makes this connection precise: for a bipartition of the chain into a left and right half at bond $n$, the entanglement entropy between the two halves is computed directly from the singular values at that cut,

$$S_n = -\sum_\alpha p_\alpha \log p_\alpha, \qquad p_\alpha = S_\alpha^2$$

where $p_\alpha$ are the (normalized) squared singular values, i.e. the eigenvalues of the reduced density matrix on either side of the cut. This is one of the most important diagnostic quantities in tensor network methods: it tells us exactly how entangled the state is across that bond, which in turn tells us how large the bond dimension $\chi$ needs to be to represent the state accurately.

In [ ]:
using Plots
using LaTeXStrings

function Calculate_SvN_entropy(state)
    
    N = length(siteinds(state))
    SvN = zeros(N)
    for b=1:N
        orthogonalize!(state, b)
        if b == 1
            U,S,V = svd(state[b], (siteind(state, b))) # There is no link to the left
        else
            U,S,V = svd(state[b], (linkind(state, b-1), siteind(state, b)))
        end
        for n=1:dim(S, 1)
            p = S[n,n]^2
            SvN[b] -= p * log(p)
        end
    end

    return SvN
end

In [ ]:
N = 20; # Number of sites
sites = siteinds("Qubit", N) # Indices for all sites
ψ = randomMPS(sites,linkdims=40);

SvN = Calculate_SvN_entropy(ψ)
plot(1:20, SvN, xlabel = L"n", ylabel = L"S_{n}", frame = :box, dpi = 200, linewidth = 2.0, label = false)

## Encoding functions and Hadamard product (element-wise multiplication)

So far, every operation we've discussed — applying operators, computing overlaps, adding states — treats the MPS as a genuine quantum state living in a linear vector space. But there's an entirely different, equally important operation: the Hadamard product, i.e., multiplying two MPS element-wise, index by index, rather than contracting them as an inner product or superposing them as a sum:

$$(\psi \odot \phi)_{s_1 s_2 \cdots s_N} = \psi_{s_1 s_2 \cdots s_N} \cdot \phi_{s_1 s_2 \cdots s_N}$$

This is not a standard quantum-mechanical operation — there's no physical "state" interpretation of multiplying two amplitudes together — but it becomes essential the moment we start using MPS to represent functions rather than quantum states. If $\psi$ encodes a function $f(x)$ and $\phi$ encodes $g(x)$ (both discretized/binarized across the same $N$ physical indices), then $\psi \odot \phi$ encodes the pointwise product $f(x)g(x)$ — directly enabling us to compute non-linear combinations of functions already stored in MPS form, such as $f(x)^2$, without ever leaving the compressed tensor-network representation.

Before explaining how to implement the Hadamard product, **let's discuss how to encode a function with qubits**. 

Consider $f(x)$ a function with $x \in [0, 1]$. The domain can be discretized through $N$ qubits into $2^N$ grid points $x_n = n/2^N$, for $n = 0, 1, \dots, 2^N - 1$.

**Example with $N = 4$:**

$$
x_n = \{0,\ 1/16,\ 2/16,\ \dots,\ 15/16\}
$$

<div align="center">

```
0        1/4       2/4       3/4        1
|----|----|----|----|----|----|----|----|
                x₅=1/4          x₉=2/4
```
</div>

Note that we can encode $x_n$ with $4$ qubits, i.e. for $x_n$ there exist $n_1, n_2, n_3, n_4 \in \{0,1\}$ such that:

$$
x_n = x_{n_1 n_2 n_3 n_4} = \frac{n_1}{2} + \frac{n_2}{2^2} + \frac{n_3}{2^3} + \frac{n_4}{2^4}, \qquad n_j = 0, 1
$$

$$
\left(\, x_{13} = x_{1100} = \frac{1}{2} + \frac{1}{4} = \frac{3}{4} \,\right)
$$

**Binary encoding table** (each column is one value $n = 0, \dots, 15$; rows are the qubit values $n_1 n_2 n_3 n_4$ — column $n=13$, i.e. $1100$, is highlighted):

| $n$   | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 | 11 | **12** | **13** | 14 | 15 |
|-------|---|---|---|---|---|---|---|---|---|---|----|----|----|----|----|----|
| $n_1$ (biggest scale) | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 1 | 1 | 1 | 1 | **1** | 1 | 1 | 1 |
| $n_2$ | 0 | 0 | 0 | 0 | 1 | 1 | 1 | 1 | 0 | 0 | 0 | 0 | **1** | 1 | 1 | 1 |
| $n_3$ | 0 | 0 | 1 | 1 | 0 | 0 | 1 | 1 | 0 | 0 | 1 | 1 | **0** | 0 | 1 | 1 |
| $n_4$ (smallest scale) | 0 | 1 | 0 | 1 | 0 | 1 | 0 | 1 | 0 | 1 | 0 | 1 | **0** | 1 | 0 | 1 |

**We can do the same with a function!**

$$
f(x) \longrightarrow \{f(x_n)\}_{n=1,\dots,N}
$$

$$
|\psi\rangle = \sum_{n_1, \dots, n_N = 0, 1} f_{n_1 n_2 \dots n_N} \, |n_1\rangle \otimes |n_2\rangle \otimes \dots \otimes |n_N\rangle
$$

Now, imagine we have two functions represented as MPS. 

- $C_{n_1, \dots, n_N} = A^{n_1} A^{n_2} \cdots A^{n_N}$ &nbsp; ($\chi_A$ — bond dimension)
- $D_{m_1, \dots, m_N} = B^{m_1} B^{m_2} \cdots B^{m_N}$ &nbsp; ($\chi_B$ — bond dimension)

Although the hadamaoughrd product is conceptually simple, for example for rank-2 tensors: 
$
A = \begin{pmatrix} 2 & -1 \\ 3 & 5 \end{pmatrix}, B = \begin{pmatrix} -2 & 7 \\ 2 & 1 \end{pmatrix} \Rightarrow A \odot B = \begin{pmatrix} -4 & -7 \\ 6 & 5 \end{pmatrix}
$

For the MPS case we have
$$
 F = C \odot D = F_{p_1 p_2 \dots p_N} = A^{p_1} A^{p_2} \cdots A^{p_N}\, B^{p_1} B^{p_2} \cdots B^{p_N}.
$$

To perform this, we need to define the Kronecker deltas $\delta_{n_j m_j p_j}$, which is just one when $n_j = m_j = p_j$.

$$
\therefore\quad F = A^{n_1} A^{n_2} \cdots A^{n_N}\, B^{m_1} B^{m_2} \cdots B^{m_N}\, \delta_{n_1 m_1 p_1} \delta_{n_2 m_2 p_2} \cdots \delta_{n_N m_N p_N}
$$

In diagrammatic notation:

<div align="center">

<img src="Figures/Hadamard product.png" width="800"/>

</div>

**The complexity scales as** $(\chi_A \chi_B)^2$ and **the bond dimension increases as** $\chi_A \chi_B$. 

In [ ]:
function Apply_Hadamard_ψ1_ψ2(ψ1, ψ2, χ, cutoff)
    sites = siteinds(ψ1)
    N = length(sites)
    had = MPO(N) 
    had .= convert(MPO, ψ1)
    for i = 1:N
        had[i] *= δ(sites[i], sites[i]', sites[i]'')
    end
    had = replaceprime(had, 2 => 0)
    return apply(had, ψ2; maxdim=χ, cutoff=cutoff)
end

To be honest, I have never used this operation in my research. It tends to appear in **non-linear** differential equations, and most of quantum mechanics consists of **linear** differential equations.

Let's take as an example $h(x) = f(x) g(x)$ with $ f(x) = \sin (x)$ and $g(x) = x e^{0.25 x}$.

In [ ]:
N = 7  # Number of qubits used for encoding the function.
x_values = range(0, 2pi, length=2^N); # Sampling grid points

f(x) = sin(x); # First function
y1 = f.(x_values); # Construction f(x)

g(x) = x*exp(0.25*x); # Second function
y2 = g.(x_values); # Construction g(x)

h(x) = f(x)*g(x);
y3 = h.(x_values); # Construction h(x)

plot(x_values, y3, frame = :box, dpi = 200, linewidth = 2.0, label = false, xlabel = "x", ylabel = L"h(x) = \sin (x) x e^{0.25 x}") # Plotting the original function

In [ ]:
sites = siteinds("Qubit", N); # Indices for all sites

cutoff = 1E-8;
χ = 2;

ψ1 = MPS(y1, sites, cutoff=cutoff, maxdim=χ);
ψ2 = MPS(y2, sites, cutoff=cutoff, maxdim=χ);
ψ3 = MPS(y3, sites, cutoff=cutoff, maxdim=χ);

@show linkdims(ψ1);
@show linkdims(ψ2);
@show linkdims(ψ3);


ψ3_Had = Apply_Hadamard_ψ1_ψ2(ψ1, ψ2, χ, cutoff)
@show linkdims(ψ3_Had);

In [ ]:
# Now let's go back to standardJulia array objects to see that we have the same information.

Big_ψ = contract(ψ3_Had) 
comb = combiner(sites)
y_recon = Array(Big_ψ*comb, combinedind(comb)) # Changing the format from ITensor to Julia array for plotting7

#If we do not want to use the combiner, we could also do this (this is how JJ does it).
# J_recon = Array(Big_ψ, s) # Changing the format from ITensor to Julia array for plotting
# y_recon = reshape(J_recon,2^N,1) # Reshaping from N-dimentional tensor to regular array

plot(x_values, y_recon, frame = :box, dpi = 200, linewidth = 2.0, label = false, xlabel = "x", ylabel = L"h(x) = \sin (x) x e^{0.25 x}", title = L"\chi = %$χ, cutoff = %$cutoff") # Plotting the reconstructed function

## Exercise: Encode $h(x) = f^{2}(x) g(x)$ with $ f(x) = \cos (x)$ and $g(x) = \sqrt{x} e^{-0.25 x}$ for $x \in [0,20]$ using 8 qubits.

In [ ]:
#TODO:










# GPU speed-up

Everything we've built so far — random ITensors, MPS construction, SVDs, operator applications — has been running on the CPU, where the cubic scaling in $\chi$ we just discussed can become a real bottleneck as bond dimensions grow. Since **these operations are fundamentally dense linear algebra (matrix multiplications, SVDs, contractions), they map naturally onto GPU hardware**, which is built precisely to accelerate this kind of computation. The core idea is straightforward: move the relevant tensors onto the GPU, perform the same operations we've already learned — contraction, SVD, apply(), MPS construction — and let the underlying library dispatch to GPU-accelerated linear algebra routines instead of CPU ones, with **no change to the tensor-network logic itself.**

In practice, the most important part is being sure about where your data lives and how much of it fits. GPUs typically have far less memory than a CPU-accessible RAM pool, so an MPS or intermediate tensor that fits comfortably on the CPU can quickly exhaust GPU memory as bond dimensions grow. Crucially, **ITensors does not clean up GPU memory for us** — unlike Julia's CPU garbage collector, which you can mostly rely on passively, GPU memory needs to be actively monitored and freed as the code runs, or you'll run out of memory well before you expect to.

**Note:** The slow first execution of Julia is independent of GPU and CPU. You need to precompile whatever you want to use on the hardware that you will use.

In [ ]:
# using Pkg 
# Pkg.add("CUDA") #You need to install CUDA if you do not have it already. Sometimes this can be annoying depending on the drivers of your GPU

using CUDA

In [ ]:
CUDA.device() #This should print your GPU. If not, something was wrong in the installation.

## Moving data to the GPU

In general we just need to use the function `cu()` or `NDTensors.cu()` to define the same ITensor object inside the GPU Memory as a CUArray. The only difference between the two functions is that NDTensors.cu() preserves the element type of the tensors, while cu() converts to single precision.  I **ALWAYS** uses `NDTensors.cu()` because **single precision can generate problems in DMRG or TEBD** algorithms: https://itensor.discourse.group/t/tebd-with-gpu-error-with-eigen/1266/5 .

In [ ]:
#woGPU
sites = siteinds("S=1/2",50)
bond_dimension = 200 #The difference between times increases with this value.

A = randomMPS(sites, bond_dimension)
O = randomMPO(sites)

@time inner(A, apply(O, A))

In [ ]:
#wGPU
A = NDTensors.cu(A)
O = NDTensors.cu(O)

@time inner(A, apply(O, A))

## How to supervise memory?

In [ ]:
CUDA.pool_status() #This is how you check the memory status of your GPU. In the past, it was memory_status(). The name can vary with the version of CUDA.

## How to clean memory?

Since ITensors doesn't automatically free GPU memory for us, we need to explicitly trigger cleanup when memory usage grows too large — especially in loops where many intermediate tensors get created and discarded (e.g. repeated SVDs, sweeps, or gate applications).

- `GC.gc(true)` forces Julia's garbage collector to run a full, deep collection, freeing any CPU-side objects (including wrapper objects pointing to GPU arrays) that are no longer referenced. This is thorough but comparatively slow, since it walks the entire object graph rather than just recently allocated memory.

- `CUDA.reclaim()` asks the CUDA memory pool to release GPU memory that has already been marked as free (typically by a prior garbage collection) back to the device. This is a much faster, GPU-side-only operation, but it only reclaims memory that Julia's GC has already flagged as unreachable — it doesn't do the reachability analysis itself.

Let's see this by repeating the previous operation; **without cleaning at some point, the GPU will be full, and the code will be slower.**

In [ ]:
sites = siteinds("S=1/2",50)
bond_dimension = 100 #The difference between times increases with this value.

A = randomMPS(sites, bond_dimension)
O = randomMPO(sites)
A = NDTensors.cu(A)
O = NDTensors.cu(O)

In [ ]:
for i = 1:20
    @time begin 
        @show i
        inner(A, apply(O, A))
        CUDA.pool_status() 

        #Cleaning:
        GC.gc(true) #Deep cleaning. Takes more time
        CUDA.reclaim() #Fast cleaning. Runs very fast.
        CUDA.pool_status() 
    end
end

In [ ]:
GC.gc(true) #Deep cleaning. Takes more time
CUDA.reclaim() #Fast cleaning. Runs very fast.

# I hope you found this material useful.

<div align="center">

<img src="Figures/Roc.jpg" width="800"/>

</div>

## Please fill out this form: https://docs.google.com/forms/d/e/1FAIpQLSenWkR35aqAo_cK5PxLxzP_lYlAGp7laQPt_kfH5_-mKpZiWA/viewform?usp=publish-editor 

<div align="center">

<img src="Figures/QR.png" width="800"/>

</div>